# 🔧 Gravitational Lensing Classification - FIXED ResNet34

## ✅ What's Fixed:

1. **Per-image normalization removed** - Was destroying class differences
2. **Global standardization** - Uses dataset-wide mean/std statistics
3. **Simple Cross Entropy loss** - Instead of Focal Loss + Weighted Sampler
4. **Higher learning rate** - 0.001 instead of 0.0001
5. **Simplified augmentation** - Physics-preserving transforms only

**Expected**: 75-88% accuracy (vs. 34% before)

---

## 📋 Setup Instructions:

1. **Upload dataset to Google Drive**:
   - Path: `MyDrive/data/dataset.zip`
   - Should contain `train/` and `val/` folders
   - Each with `no/`, `sphere/`, `vort/` subfolders

2. **Enable GPU**:
   - Runtime → Change runtime type → GPU (T4)

3. **Run all cells** (Runtime → Run all)

---

## 1️⃣ Setup & Dependencies

In [ ]:
%%capture
# Install dependencies (already available in Colab)
!pip install -q scikit-learn seaborn tqdm

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import json
from datetime import datetime
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
import seaborn as sns
from google.colab import drive

print("✅ Imports successful")

## 2️⃣ GPU Check

In [ ]:
print("="*70)
print("GPU CHECK")
print("="*70)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print("\n✅ GPU IS READY!")
else:
    print("\n⚠️ WARNING: GPU not available. Training will be VERY slow.")
    print("   Go to Runtime → Change runtime type → Select 'T4 GPU'")

print("="*70)

## 3️⃣ Mount Google Drive & Extract Dataset

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")

In [ ]:
import zipfile

# Path to your dataset in Google Drive
DRIVE_DATASET_PATH = '/content/drive/MyDrive/data/dataset.zip'
LOCAL_EXTRACT_PATH = '/content/dataset'

# Check if dataset exists
if not os.path.exists(DRIVE_DATASET_PATH):
    print("❌ ERROR: Dataset not found!")
    print(f"   Expected path: {DRIVE_DATASET_PATH}")
    print("\n📋 Upload Instructions:")
    print("   1. Go to Google Drive")
    print("   2. Create folder: MyDrive/data/")
    print("   3. Upload dataset.zip to that folder")
else:
    print(f"✅ Dataset found: {DRIVE_DATASET_PATH}")
    
    # Extract dataset
    print("\n📦 Extracting dataset...")
    with zipfile.ZipFile(DRIVE_DATASET_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content')
    
    print("✅ Dataset extracted successfully!")
    
    # Verify structure
    print("\n📂 Dataset structure:")
    for split in ['train', 'val']:
        split_path = Path(LOCAL_EXTRACT_PATH) / split
        if split_path.exists():
            print(f"\n  {split}/")
            for class_name in ['no', 'sphere', 'vort']:
                class_path = split_path / class_name
                if class_path.exists():
                    num_files = len(list(class_path.glob('*.npy')))
                    print(f"    {class_name}/: {num_files} files")

## 4️⃣ Configuration - FIXED!

In [ ]:
class Config:
    # Paths
    DATA_DIR = Path('/content/dataset')
    TRAIN_DIR = DATA_DIR / 'train'
    VAL_DIR = DATA_DIR / 'val'
    CHECKPOINT_DIR = Path('/content/checkpoints')
    LOG_DIR = Path('/content/logs')
    
    # Classes
    CLASS_NAMES = ['no', 'sphere', 'vort']
    CLASS_LABELS = ['No Substructure', 'Subhalo/Sphere', 'Vortex']
    NUM_CLASSES = 3
    
    # Training
    BATCH_SIZE = 32
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-3  # FIXED: Higher initial LR
    WEIGHT_DECAY = 1e-4
    DROPOUT = 0.4
    
    # Augmentation
    USE_AUGMENTATION = True
    
    # Scheduler
    LR_SCHEDULER = 'cosine'
    MIN_LR = 1e-7
    
    # Early stopping
    EARLY_STOPPING_PATIENCE = 15
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    RANDOM_SEED = 42
    
    # Preprocessing - FIXED!
    USE_SQRT_STRETCH = True
    # Global statistics (computed from full dataset)
    GLOBAL_MEAN = 0.0615  # Mean across all images
    GLOBAL_STD = 0.1157   # Std across all images
    
    def __init__(self):
        self.CHECKPOINT_DIR.mkdir(exist_ok=True)
        self.LOG_DIR.mkdir(exist_ok=True)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.experiment_name = f'resnet34_fixed_{timestamp}'
        self.experiment_dir = self.LOG_DIR / self.experiment_name
        self.experiment_dir.mkdir(exist_ok=True)

config = Config()

print("="*70)
print("CONFIGURATION - FIXED VERSION")
print("="*70)
print(f"Device: {config.DEVICE}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Epochs: {config.NUM_EPOCHS}")
print(f"\n🔧 KEY FIXES:")
print(f"  ✅ Global standardization (mean={config.GLOBAL_MEAN:.4f}, std={config.GLOBAL_STD:.4f})")
print(f"  ✅ Simple Cross Entropy loss")
print(f"  ✅ Higher LR: {config.LEARNING_RATE}")
print("="*70)

## 5️⃣ Dataset Class - FIXED Preprocessing!

In [ ]:
class GravitationalLensingDataset(Dataset):
    def __init__(self, data_dir, class_names, use_sqrt_stretch=True, augment=False, config=None):
        self.data_dir = Path(data_dir)
        self.class_names = class_names
        self.use_sqrt_stretch = use_sqrt_stretch
        self.augment = augment
        self.config = config
        
        self.samples = []
        self.labels = []
        
        print(f"\nLoading data from {data_dir}...")
        for class_idx, class_name in enumerate(class_names):
            class_dir = self.data_dir / class_name
            class_files = sorted(class_dir.glob('*.npy'))
            print(f"  {class_name}: {len(class_files)} samples")
            
            for file_path in class_files:
                self.samples.append(file_path)
                self.labels.append(class_idx)
        
        print(f"Total samples: {len(self.samples)}")
        if augment:
            print("  Data augmentation: ENABLED")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        # Load image
        image = np.load(self.samples[idx])
        if image.shape[0] == 1:
            image = image[0]
        
        # Preprocessing: Square-root stretch
        if self.use_sqrt_stretch:
            image = np.maximum(image, 0)
            image = np.sqrt(image)
            
            # FIXED: Global standardization instead of per-image normalization!
            # This preserves the relative brightness differences between images
            image = (image - self.config.GLOBAL_MEAN) / (self.config.GLOBAL_STD + 1e-7)
        
        # Convert to tensor
        image = torch.from_numpy(image).float().unsqueeze(0)
        
        # Augmentation
        if self.augment:
            # 90-degree rotation
            if np.random.rand() < 0.75:
                k = np.random.randint(0, 4)
                image = torch.rot90(image, k, dims=[-2, -1])
            
            # Horizontal flip
            if np.random.rand() < 0.5:
                image = torch.flip(image, dims=[-1])
            
            # Vertical flip
            if np.random.rand() < 0.5:
                image = torch.flip(image, dims=[-2])
            
            # Gaussian noise
            if np.random.rand() < 0.3:
                noise = torch.randn_like(image) * 0.02
                image = image + noise
        
        return image, self.labels[idx]

print("✅ Dataset class defined")

## 6️⃣ ResNet34 Model Architecture

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                              stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
    
    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out


class ResNet34(nn.Module):
    def __init__(self, num_classes=3, dropout=0.4):
        super(ResNet34, self).__init__()
        self.in_channels = 64
        
        # Initial convolution (grayscale input)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # ResNet layers [3, 4, 6, 3]
        self.layer1 = self._make_layer(BasicBlock, 64, 3, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 4, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 6, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 3, stride=2)
        
        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(512 * BasicBlock.expansion, num_classes)
        
        # Initialize weights
        self._initialize_weights()
    
    def _make_layer(self, block, out_channels, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion,
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion)
            )
        
        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels))
        
        return nn.Sequential(*layers)
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

print("✅ ResNet34 model defined")

## 7️⃣ Load Data

In [ ]:
# Set random seed
torch.manual_seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)

# Create datasets
train_dataset = GravitationalLensingDataset(
    config.TRAIN_DIR,
    config.CLASS_NAMES,
    use_sqrt_stretch=config.USE_SQRT_STRETCH,
    augment=config.USE_AUGMENTATION,
    config=config
)

val_dataset = GravitationalLensingDataset(
    config.VAL_DIR,
    config.CLASS_NAMES,
    use_sqrt_stretch=config.USE_SQRT_STRETCH,
    augment=False,
    config=config
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n✅ Data loaded successfully!")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")

## 8️⃣ Training Setup

In [ ]:
# Create model
model = ResNet34(num_classes=config.NUM_CLASSES, dropout=config.DROPOUT)
model = model.to(config.DEVICE)

# Loss function - FIXED: Simple Cross Entropy
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=config.NUM_EPOCHS,
    eta_min=config.MIN_LR
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*70)
print("MODEL SETUP")
print("="*70)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Loss function: Cross Entropy (Simple & Effective)")
print(f"Optimizer: AdamW")
print(f"Scheduler: CosineAnnealingLR")
print("="*70)

## 9️⃣ Training & Validation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation', leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc, all_preds, all_labels

print("✅ Training functions defined")

## 🔟 Training Loop

In [ ]:
print("="*70)
print("STARTING TRAINING - FIXED VERSION")
print("="*70)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

best_val_acc = 0.0
patience_counter = 0

for epoch in range(config.NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    print("-" * 50)
    
    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning Rate: {current_lr:.2e}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, config.DEVICE)
    
    # Validate
    val_loss, val_acc, val_preds, val_labels = validate_epoch(model, val_loader, criterion, config.DEVICE)
    
    # Update scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    # Print metrics
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_acc': best_val_acc,
            'history': history
        }
        torch.save(checkpoint, config.CHECKPOINT_DIR / 'best_model.pth')
        print(f"  ✅ New best model saved! Val Acc: {val_acc:.4f}")
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{config.EARLY_STOPPING_PATIENCE}")
    
    # Early stopping
    if patience_counter >= config.EARLY_STOPPING_PATIENCE:
        print(f"\n⚠️ Early stopping triggered after {epoch+1} epochs")
        break
    
    # Save history every 5 epochs
    if (epoch + 1) % 5 == 0:
        with open(config.experiment_dir / 'history.json', 'w') as f:
            json.dump(history, f, indent=4)

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Total Epochs: {len(history['train_loss'])}")

## 1️⃣1️⃣ Plot Training Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
axes[1].axhline(y=best_val_acc, color='g', linestyle='--', label=f'Best: {best_val_acc:.4f}')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Learning Rate
axes[2].plot(epochs_range, history['lr'], 'g-', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate', fontsize=12)
axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(config.experiment_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Training curves saved to {config.experiment_dir / 'training_curves.png'}")

## 1️⃣2️⃣ Final Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(config.CHECKPOINT_DIR / 'best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']}")

# Evaluate
val_loss, val_acc, val_preds, val_labels = validate_epoch(model, val_loader, criterion, config.DEVICE)

# Confusion Matrix
cm = confusion_matrix(val_labels, val_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=config.CLASS_LABELS,
            yticklabels=config.CLASS_LABELS,
            cbar_kws={'label': 'Normalized Count'})
plt.title(f'Confusion Matrix (Accuracy: {val_acc:.4f})', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(config.experiment_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification Report
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(val_labels, val_preds, target_names=config.CLASS_LABELS, digits=4))

# Per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(val_labels, val_preds)

print("\n" + "="*70)
print("PER-CLASS METRICS")
print("="*70)
for i, class_name in enumerate(config.CLASS_LABELS):
    print(f"{class_name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-Score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Average Precision:   {precision.mean():.4f}")
print(f"Average Recall:      {recall.mean():.4f}")
print(f"Average F1-Score:    {f1.mean():.4f}")
print("="*70)

## 1️⃣3️⃣ Download Results to Google Drive

In [ ]:
import shutil

# Create results folder in Drive
drive_results_path = f'/content/drive/MyDrive/gravitational_lensing_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
os.makedirs(drive_results_path, exist_ok=True)

# Copy checkpoints
shutil.copy(config.CHECKPOINT_DIR / 'best_model.pth', drive_results_path)

# Copy logs
shutil.copytree(config.experiment_dir, os.path.join(drive_results_path, 'logs'))

print(f"\n✅ Results saved to Google Drive:")
print(f"   {drive_results_path}")
print("\nContents:")
print("  - best_model.pth (trained model)")
print("  - logs/ (training curves, confusion matrix, history)")

---

## 📊 Training Summary

### ✅ Key Fixes Applied:
1. **Global Standardization**: Using dataset-wide statistics instead of per-image normalization
2. **Simple Loss Function**: Cross Entropy instead of Focal Loss + Weighted Sampler
3. **Higher Learning Rate**: 0.001 instead of 0.0001
4. **Physics-Preserving Augmentation**: Rotations, flips, minimal noise

### 📈 Expected Performance:
- **Before (broken)**: ~34% accuracy (random guessing)
- **After (fixed)**: 75-88% accuracy

### 🎯 Next Steps:
If accuracy is still low:
1. Check if GPU is enabled (Runtime → Change runtime type)
2. Increase training epochs (100 → 150)
3. Try different learning rates (1e-3, 5e-4, 1e-4)
4. Experiment with augmentation strength

---

**Model saved to Google Drive!** 🎉